# 🔀 Multiprocessing

## 📖 Introduction

Ce chapitre est la suite du chapitre sur le multithreading.

Maintenant que vous connaissez le GIL et l'utilisation des threads, voyons comment Python peut exécuter du code en parallèle.

<img src='files/multithreading_vs_multiprocessing.png' alt='Multithreading vs Multiprocessing diagram' width='600' source="miro.medium.com">

## 📦 Le module Multiprocessing

Le module `multiprocessing` de Python permet de créer plusieurs processus, chacun avec son propre interpréteur Python et son propre espace mémoire. C'est particulièrement utile pour les tâches CPU-bound qui bénéficient du parallélisme.

### 🎯 Concurrent.futures

Vous vous souvenez du module `concurrent.futures`, qui fournit une interface haut niveau pour exécuter des appels de façon asynchrone ?

Dans le chapitre précédent, nous avons utilisé `concurrent.futures.ThreadPoolExecutor` pour exécuter des tâches de manière concurrente avec des threads.

Ici, nous allons utiliser `concurrent.futures.ProcessPoolExecutor` pour exécuter des tâches de manière concurrente avec des processus.

La syntaxe est très proche de celle utilisée avec les threads, donc on peut facilement passer de l'un à l'autre.

Reprenons le même code que dans le chapitre précédent et adaptons-le pour utiliser `ProcessPoolExecutor` à la place de `ThreadPoolExecutor`.

Cette fois, nous utiliserons aussi la fonction `os.getpid()` pour afficher l'identifiant du processus de chaque tâche, afin de voir quel processus exécute quoi.

In [ ]:
# With ThreadPoolExecutor and os.getpid()

import threading
import concurrent.futures
import time
import os

print(f"Main process {os.getpid()}")

def task(n):
    print(f"{" " * n}Task {n} starting on thread {threading.current_thread().name}, process {os.getpid()}")
    time.sleep(n)
    print(f" {" " * n}Task {n} completed on thread {threading.current_thread().name}, process {os.getpid()}")
    return n * n

with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(task, [1, 2, 3, 4, 5])

print("Results:", list(results))
print("Main thread finished.")

In [ ]:
# Now let's try with ProcessPoolExecutor.

import threading
import concurrent.futures
import time
import os

print(f"Main process {os.getpid()}")
def task(n):
    print(f"{" " * n}Task {n} starting on process {threading.current_thread().name}, process {os.getpid()}")
    time.sleep(n)
    print(f" {" " * n}Task {n} completed on process {threading.current_thread().name}, process {os.getpid()}")
    return n * n
with concurrent.futures.ProcessPoolExecutor(max_workers=3) as executor:
    results = executor.map(task, [1, 2, 3, 4, 5])
print("Results:", list(results))


- Avec `ThreadPoolExecutor`, l'identifiant de processus est le même pour toutes les tâches, car elles s'exécutent dans un seul et même processus.

- Avec `ProcessPoolExecutor`, chaque tâche s'exécute dans un processus OS séparé, donc l'identifiant de processus varie selon les tâches. Si les sorties semblent peu lisibles, c'est normal : Jupyter Notebook n'est pas idéal pour afficher proprement les sorties de multiples processus.

## ⚡ Comparaison de vitesse sur une tâche CPU-bound

Comparons les performances entre exécution séquentielle, threads et processus pour une tâche liée au CPU.

In [ ]:
# Without any threading or processing

import time

def cpu_bound_task(n):
    print(f"Starting task {n} in process {os.getpid()}")
    count = 0
    for i in range(10**7):
        count += i % n
    print(f"Completed task {n} in process {os.getpid()}")
    return count

start = time.time()

for n in range(1, 6):
    cpu_bound_task(n)
end = time.time()
print(f"Total time without any executor: {end - start} seconds")


In [ ]:
# With Multithreading (ThreadPoolExecutor)

import time

def cpu_bound_task(n):
    print(f"Starting task {n} in process {os.getpid()}")
    count = 0
    for i in range(10**7):
        count += i % n
    print(f"Completed task {n} in process {os.getpid()}")
    return count

start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    executor.map(cpu_bound_task, range(1, 6))
    
end = time.time()
print(f"Total time ThreadPoolExecutor: {end - start} seconds")

In [ ]:
# With Multiprocessing (ProcessPoolExecutor)

import time

def cpu_bound_task(n):
    print(f"Starting task {n} in process {os.getpid()}")
    count = 0
    for i in range(10**7):
        count += i % n
    print(f"Completed task {n} in process {os.getpid()}")
    return count

start = time.time()

with concurrent.futures.ProcessPoolExecutor(max_workers=6) as executor:
    executor.map(cpu_bound_task, range(1, 6))
    
end = time.time()
print(f"Total time with ProcessPoolExecutor: {end - start} seconds")


### 📊 Conclusion

- Pour une tâche CPU-bound, utiliser un seul processus est équivalent, voire parfois plus rapide, que d'utiliser des threads à cause du GIL.

- Avec `ProcessPoolExecutor`, on observe en général un gain significatif, car chaque processus peut tourner sur un cœur CPU distinct, en contournant la limitation du GIL.

### 💪 Exercice

CPython existe maintenant en deux variantes : une avec GIL et une sans GIL (souvent appelée Gilectomy).

Essayez de reproduire cette expérience avec la version sans GIL et observez comment les résultats évoluent.

https://medium.com/sdg-group/exploring-pythons-gil-single-multithreading-vs-multiprocessing-and-the-impact-of-gil-removal-ee8b6dd610f4

## 🚀 Autres façons d'exécuter du code en parallèle

La bibliothèque `multiprocessing` n'est pas la seule manière d'exécuter du code en parallèle en Python. Certaines bibliothèques proposent leurs propres approches.

### 🔢 NumPy / SciPy

NumPy est rapide car les calculs lourds sont faits en C/Fortran compilé (donc hors GIL). Ces bibliothèques bas niveau (BLAS, LAPACK, MKL, OpenBLAS) utilisent souvent plusieurs cœurs CPU automatiquement.

Si vous surveillez votre CPU pendant ce code, vous verrez tous les cœurs utilisés. Mais ce n'est pas du multiprocessing du point de vue Python : le parallélisme est géré par les bibliothèques sous-jacentes.

Dans ce cas, NumPy agit comme une **exécution mono-processus, multi-threads, multi-cœurs** (parallélisme). Techniquement, il n'y a donc pas de multiprocessing Python explicite.

In [ ]:
import numpy as np
# np.show_config() # Check if NumPy is linked against a multi-threaded BLAS implementation

A = np.random.rand(5000, 5000) # If you have a powerful computer,
B = np.random.rand(5000, 5000) # you can increase the size of the matrices.

# Matrix multiplication
C = A @ B  # Check your CPU usage during this operation !

### ⚡ Numba

Numba est un compilateur just-in-time pour Python qui traduit un sous-ensemble de Python et NumPy en code machine rapide. Il peut paralléliser automatiquement certaines opérations sur plusieurs cœurs CPU.

### 🤖 PyTorch / TensorFlow / JAX

Ces frameworks sont pensés pour le parallélisme. Ils s'appuient sur des backends C++, relâchent le GIL et exploitent CPU et GPU.

### 📊 Dask

Dask est une bibliothèque de calcul parallèle flexible pour l'analytics. Elle permet de passer d'une machine unique à un cluster et de paralléliser facilement des opérations NumPy, Pandas, etc.

### 🐻‍❄️ Polars

Polars est une bibliothèque DataFrame rapide implémentée en Rust. Elle est conçue pour la performance et peut exploiter plusieurs cœurs CPU. Son principal inconvénient est une API différente de Pandas, qui peut nécessiter d'adapter le code.

### ⚡ PySpark

PySpark est l'API Python d'Apache Spark, un framework de calcul distribué. Il permet de traiter de gros volumes de données en parallèle sur un cluster. Sur une seule machine, il peut aussi utiliser plusieurs cœurs CPU, mais avec davantage d'overhead que Dask ou Polars pour de petits jeux de données. Donc, si vous ne prévoyez pas de cluster à court terme, Dask ou Polars sont souvent préférables.

### 🦀 Code Rust dans Python

Si vous avez besoin de performances extrêmes et de parallélisme, vous pouvez écrire les parties critiques en Rust puis les appeler depuis Python avec des bibliothèques comme `PyO3`. Rust offre un excellent support de la concurrence et du parallélisme.